In [21]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# ── Hyperparameters & config ─────────────────────────────────────────
TEST_SIZE      = 0.2      # fraction of data held out for testing
RANDOM_SEED    = 42

D_MODEL        = 64       # smaller model → less overfitting capacity
NHEAD          = 4        # number of attention heads
NUM_LAYERS     = 2        # single layer is enough for 40 samples
DROPOUT        = 0.25      # stronger dropout

BATCH_SIZE     = 16
NUM_EPOCHS     = 250      # more epochs; early stopping will cut it short
LR             = 1e-3
WEIGHT_DECAY   = 2e-3     # stronger weight decay
LR_STEP        = 30       # decay LR every N epochs
LR_GAMMA       = 0.5      # LR decay factor

PATIENCE       = 1000       # early stopping patience (epochs without improvement)

In [40]:
# Load data
df = pd.read_csv('processed_data/patient_data.csv', index_col=0)

# Drop metadata and duplicate columns
drop_cols = ['Code', 'BrainSegVolNotVent_y', 'eTIV_x', 'eTIV_y']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Encode target and Sex
df['label'] = (df['Group'] == 'PD').astype(int)
df['Sex'] = (df['Sex'] == 'M').astype(int)
df = df.drop(columns=['Group'])

# Features and labels
feature_cols = [c for c in df.columns if c != 'label']
X = df[feature_cols].values.astype('float32')
y = df['label'].values.astype('float32')

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Features: {len(feature_cols)}')
print(f'Label distribution — Train: {y_train.sum():.0f} PD / {(1-y_train).sum():.0f} HC')

Train: (40, 137), Test: (11, 137)
Features: 137
Label distribution — Train: 16 PD / 24 HC


/var/folders/6y/mzx9s2x10k7bvzyhx74mw82c0000gn/T/ipykernel_20399/3484941983.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['label'] = (df['Group'] == 'PD').astype(int)


In [41]:
class BrainDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = BrainDataset(X_train, y_train)
test_dataset = BrainDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

Train batches: 3, Test batches: 1


In [42]:
class TabularTransformer(nn.Module):
    def __init__(self, num_features, d_model=D_MODEL, nhead=NHEAD,
                 num_layers=NUM_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.feature_proj = nn.Linear(1, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dropout=dropout,
            dim_feedforward=d_model * 4, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x):
        B, F = x.shape
        x = self.feature_proj(x.unsqueeze(-1))
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = self.transformer(x)
        return self.head(x[:, 0, :]).squeeze(-1)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TabularTransformer(num_features=len(feature_cols)).to(device)
print(model)
print(f'\nParameters: {sum(p.numel() for p in model.parameters()):,}')

TabularTransformer(
  (feature_proj): Linear(in_features=1, out_features=128, bias=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.25, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.25, inplace=False)
        (dropout2): Dropout(p=0.25, inplace=False)
      )
    )
  )
  (head): Linear(in_features=128, out_features=1, bias=True)
)

Parameters: 397,057


In [43]:
criterion  = nn.BCEWithLogitsLoss()
optimizer  = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler  = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP, gamma=LR_GAMMA)

best_loss      = float('inf')
best_state     = None
patience_count = 0

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    avg_loss = total_loss / len(train_loader)

    # Early stopping
    if avg_loss < best_loss:
        best_loss  = avg_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1}  best loss: {best_loss:.4f}')
            break

    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1}/{NUM_EPOCHS}  loss: {avg_loss:.4f}')

model.load_state_dict(best_state)
print(f'\nRestored best model (loss {best_loss:.4f})')

Epoch 20/400  loss: 0.7027
Epoch 40/400  loss: 0.6674
Epoch 60/400  loss: 0.6245
Epoch 80/400  loss: 0.6658
Epoch 100/400  loss: 0.6955
Epoch 120/400  loss: 0.6382
Epoch 140/400  loss: 0.6371
Epoch 160/400  loss: 0.6662
Epoch 180/400  loss: 0.6480
Epoch 200/400  loss: 0.6484
Epoch 220/400  loss: 0.6691
Epoch 240/400  loss: 0.6819
Epoch 260/400  loss: 0.6375
Epoch 280/400  loss: 0.6412
Epoch 300/400  loss: 0.6904
Epoch 320/400  loss: 0.6667
Epoch 340/400  loss: 0.6668
Epoch 360/400  loss: 0.6500
Epoch 380/400  loss: 0.6301
Epoch 400/400  loss: 0.6535

Restored best model (loss 0.6016)


In [44]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        logits = model(X_batch)
        preds = (torch.sigmoid(logits) > 0.5).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print(f"Accuracy: {accuracy_score(all_labels, all_preds):.3f}")
print()
print(classification_report(all_labels, all_preds, target_names=['HC', 'PD']))
print("Confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(
    confusion_matrix(all_labels, all_preds),
    index=['True HC', 'True PD'],
    columns=['Pred HC', 'Pred PD']
))

Accuracy: 0.727

              precision    recall  f1-score   support

          HC       0.70      1.00      0.82         7
          PD       1.00      0.25      0.40         4

    accuracy                           0.73        11
   macro avg       0.85      0.62      0.61        11
weighted avg       0.81      0.73      0.67        11

Confusion matrix (rows=true, cols=pred):
         Pred HC  Pred PD
True HC        7        0
True PD        3        1
